In [2]:
import numpy as np
import pandas as pd
import random
from attitude import q_to_e, e_to_q, q_to_DCM, DCM_to_q, DCM, DCM_to_e, e_to_DCM
data_path = "./hygdata_v42.csv"
data = pd.read_csv(data_path)
data.drop_duplicates(subset="hr", inplace=True)
data = data[1:]
mask = data["hr"].notna()

In [47]:
from camera import Camera
cam = Camera(data[mask])
cam.id_point(753, 0)
# cam.rand_point()
focal = 671
res = [360, 360]
coords = cam.create_centroids(focal, res, None, None)
unique_hr_ids = sorted(set(hr for hr in data["hr"]))
hr_to_idx = {hr_id: i for i, hr_id in enumerate(unique_hr_ids)}
idx_to_hr = {i: hr_id for hr_id, i in hr_to_idx.items()} 

In [48]:
from model import star_tracker_v1
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = star_tracker_v1(25, 9029, 128, 128)
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))

<All keys matched successfully>

In [49]:
def identify_stars(coords, res, model, n_bin, idx_to_hr, n_candidates):
    center = np.array([res[1] / 2, res[0] / 2])
    center_distance = np.linalg.norm((coords[["px", "py"]] - center), axis = 1)
    closest_stars = np.argpartition(center_distance, n_candidates - 1)[:n_candidates]

    max_radi = np.sqrt((res[0]**2) + (res[1]**2)) / 2
    bins = np.linspace(0, max_radi, n_bin + 1)

    pred_ids = []

    model.eval()

    with torch.inference_mode():
        for candidate_idx in closest_stars:
            guide_star = coords.iloc[candidate_idx][["px", "py"]]
            
            coords_copy = coords[["px", "py"]].copy().to_numpy()
            coords_copy  = np.delete(coords_copy, candidate_idx, axis=0)
            
            guide_vector = (center - guide_star).to_numpy()
            coords_copy += guide_vector
    
            visible = ((coords_copy[:, 0] >= 0) & (coords_copy[:, 0] < res[0]) &
                       (coords_copy[:, 1] >= 0) & (coords_copy[:, 1] < res[1]))
            coords_copy = coords_copy[visible]
    
            distances = np.linalg.norm(coords_copy - center, axis=1)
            histo = np.histogram(distances, bins)
            
            result = histo[0].astype(np.float32)
            result /= max(result.sum(), 1.0)
    
            x = torch.from_numpy(result).unsqueeze(0)
            pred_id = model(x).argmax(dim=1).item()
            pred_ids.append(idx_to_hr[pred_id])

    return closest_stars, pred_ids

In [50]:
test = identify_stars(coords, res, model, 25, idx_to_hr, 3)

In [51]:
filtered = cam.data.loc[mask & data["hr"].isin(test[1])]
filtered[["hr", "ux", "uy", "uz"]]

,hr,ux,uy,uz
11991,751.0,0.773077,0.620842,0.130024
12082,753.0,0.771316,0.625054,0.119909
12115,766.0,0.768491,0.625564,0.134505


In [52]:
cam_star = coords[coords["hr"].isin(test[1])].copy()

# camera is x
focal = 671
cx = 180
cy = 180
cam_star["px"] = (cam_star["px"] - cx) / focal # y coordinate
cam_star["py"] = (cam_star["py"] - cy) / focal # z coordinate
cam_star = cam_star.rename(columns={"px" : "y", "py" : "z", })
cam_star["x"] = 1.0
cam_star.loc[:, ["y", "z", "x"]] /= np.linalg.norm(cam_star[["y", "z", "x"]].to_numpy(dtype=float), axis = 1, keepdims=True)

In [53]:
stars = pd.merge(filtered[["hr", "ux", "uy", "uz"]], cam_star, on = "hr", how = "outer").copy()

In [54]:
stars

,hr,ux,uy,uz,y,z,x
0,751.0,0.773077,0.620842,0.130024,-4.380456e-03,1.019586e-02,0.999938
1,753.0,0.771316,0.625054,0.119909,6.606526e-07,1.455684e-08,1.000000
2,766.0,0.768491,0.625564,0.134505,2.175195e-03,1.471560e-02,0.999889


In [55]:
#bi array of camera star vectors
#ri array of reference celestial star vectors
#assumes they're all normalized
def davenportq(bi, ri, weights=1.0):
    B = a * np.matmul(bi, ri.T)
    S = B + B.T
    z = np.array([B[1][2] - B[2][1], 
                  B[2][0] - B[0][2], 
                  B[0][1] - B[1][0]])
    
    weights = np.broadcast_to(np.asarray(weights, dtype=float), (len(bi),))

    B = (weights[:, None] * bi).T @ ri
    sigma = np.trace(B)
    S = B + B.T
    z = np.array([
        B[1, 2] - B[2, 1],
        B[2, 0] - B[0, 2],
        B[0, 1] - B[1, 0],
    ])

    K = np.block([
        [np.array([[sigma]]), z[None, :]],
        [z[:, None], S - sigma * np.eye(3)],
    ])

    eigenvalues, eigenvectors = np.linalg.eigh(K)
    q = eigenvectors[:, np.argmax(eigenvalues)]

    # q and -q describe the same rotation; this makes output deterministic.
    if q[0] < 0:
        q = -q

    return q / np.linalg.norm(q)


ri = stars.loc[:, ["ux", "uy", "uz"]].to_numpy()
bi = stars.loc[:, ["x", "y", "z"]].to_numpy()
a = 1
q = davenportq(bi, ri, 1)

R = q_to_DCM(q)
test = ri @ R.T
test /= np.linalg.norm(test, axis = 1, keepdims=True)
boresight_celestial = R.T @ np.array([1.0, 0.0, 0.0])
boresight_celestial /= np.linalg.norm(boresight_celestial)

ux, uy, uz = boresight_celestial
ra = np.degrees(np.arctan2(uy, ux)) % 360
dec = np.degrees(np.arcsin(np.clip(uz, -1, 1)))
ra /= 15
print(test)
print(boresight_celestial)
print([ra, dec])


[[ 9.99938426e-01 -4.38045567e-03  1.01958568e-02]
 [ 1.00000000e+00  6.60652617e-07  1.45568378e-08]
 [ 9.99889354e-01  2.17519520e-03  1.47156045e-02]]
[0.77131673 0.62505381 0.11990933]
[2.601356999999992, 6.88686999999999]


In [58]:
mask = data["hr"] == 753

In [59]:
data[mask]

,id,hip,hd,hr,gl,bf,proper,ra,dec,dist,...,bayer,flam,con,comp,comp_primary,base,lum,var,var_min,var_max
12082,12082,12114.0,16160.0,753.0,Gl 105A,NaN,268 G. Cet,2.601357,6.88687,7.1803,...,NaN,NaN,Cet,1,12082,Gl 105,0.21697,NaN,NaN,NaN
